# Game phase transition detection

**Recipe 87** · Level 3 (Intermediate) · Decision type: `Choice`

Sources:

- S02: [Decision primitives](https://docs.typesafe.ai/primitives)
- S03: [Confidence](https://docs.typesafe.ai/confidence)
- S09: [Images](https://docs.typesafe.ai/images)
- S11: [Models](https://docs.typesafe.ai/models)
- S12: [Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Build an observer that records a game's visible phase history from ordered screenshots. Each sampled frame gets a six-option `Choice`; Python collapses repeated phases, checks an allowed transition graph and records only a permitted history. A combat → menu → combat sequence retains both changes even though its endpoints match.

The images are original, synthetic **Lantern Trail** interface drawings made for this lesson. They are not captures from a shipped or proprietary game. The responses are authored probabilities, including a deliberately wrong answer. These fixtures teach a pipeline; they do not measure Jev's recognition quality.

A [review](../../docs/glossary.md#review) is an unresolved observation sent for human inspection, not an accepted phase. All history updates here are simulated: no game input, browser control or external system is connected.

## Setup and run mode

Run from this recipe folder. Offline replay needs no key or network. `images=True` enables the experimental image path when live mode is deliberately selected; `jev-1.14.0` pins the requested model ([S09](https://docs.typesafe.ai/images), [S11](https://docs.typesafe.ai/models)). Video is represented by extracted still images; no native video API is used.

There are 14 validation clips, 14 held-out clips, and two unscored demos. The local reader caps logical requests at 24 and caches the complete image-bound request. Six clips fail capture checks before inference. Live transport attempts, including retries, have a separate backend budget; see the README for recording.

In [ ]:
import json
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_outcomes,
    select_confidence_threshold,
)
from jev_cookbook.fixtures import load_inputs, load_labels, select_split
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

apply_style()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=Path("fixtures/responses.json"), images=True, model="jev-1.14.0")
reader = helpers.PhaseReader(backend, Path("."))
questions = helpers.build_questions()
offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
selection = " (a selection step, not a reported result)"
run_header(87, "Game phase transition detection", backend=backend, n_examples=28)

## The state

Python validates the supplied capture order before sampling first, middle and last frames. It refuses duplicate or reversed timestamps, a latest frame older than one second, and sampled gaps outside 0.4–2 seconds. These fixed limits define this small observation window; they are engineering assumptions, not measured optima. Sorting an out-of-order input would hide a capture problem, so the reader rejects it.

Only three opaque image references, their relative timestamps, and task instructions reach inference. File paths, clip identifiers, split names, drawing variants and expected answers remain local. The [replay key](../../docs/glossary.md#replay-key), a hash identifying an exact request, includes the referenced image bytes. The demo starts with five captures and shows which three Python chose.

In [ ]:
demos = {e.id: e for e in select_split(examples, "demo")}
demo = demos["d01"]
state, assets = helpers.build_state(demo.fields, Path("."))
print("Captured timestamps:", [f["timestamp_seconds"] for f in demo.fields["frames"]])
print("Selected relative timestamps:", [f["timestamp_seconds"] for f in state["frames"]])
print("State keys:", sorted(state), "| asset count:", len(assets))


def show_clip(example):
    selected = helpers.sample_frames(example.fields)
    sheet = Image.new("RGB", (1440, 395), "#FEFEFE")
    draw = ImageDraw.Draw(sheet)
    font = ImageFont.load_default(size=19)
    start = selected[0]["timestamp_seconds"]
    for i, frame in enumerate(selected):
        draw.text(
            (i * 480 + 12, 8),
            f"Frame {i} / +{frame['timestamp_seconds'] - start:.1f}s",
            fill="#1E1E1E",
            font=font,
        )
        with Image.open(frame["path"]) as image:
            sheet.paste(image.convert("RGB"), (i * 480, 35))
    display(sheet)


show_clip(demo)

## The questions

The three questions independently identify the foreground phase of frames 0, 1 and 2, so they share one request and the same chronological context. Python supplies `combat`, `menu`, `map`, `rewards`, `other` and `uncertain`; the model does not invent transition labels. `other` is a clear unsupported screen; `uncertain` is insufficient visual evidence. Both require review for this observer.

A reward word in chat does not make a combat screen a reward screen. Questions ask about layout and controls, and screenshot text is treated as image content. This instruction is a testable design choice, not a guarantee against misleading pixels.

[Confidence](../../docs/glossary.md#confidence) is the top probability rescaled from the six-option uniform probability: `(p_max - 1/6) / (1 - 1/6)` ([S03](https://docs.typesafe.ai/confidence)). Python uses the minimum of the three confidences as a conservative gating score; it is not the probability that the whole sequence is correct.

In [ ]:
for name, question in questions.items():
    print(name, question.instructions)
print("Fixed phase options:")
for option, description in questions["phase_0"].criteria.items():
    print(f"  {option}: {description}")

## One answer up close

The typed answers expose every supplied probability and their [provenance](../../docs/glossary.md#provenance), the record of whether an answer was authored or captured from inference. Python then derives the ordered event string from the three chosen phases. A second demo has an incomplete scene fade in the middle; its uncertainty must survive even when both endpoints are recognizable.

In [ ]:
demo_observation = reader.observe(demo)
for i in range(3):
    print(f"Frame {i}")
    show_answer(demo_observation.result[f"phase_{i}"])
print("Derived observed event:", demo_observation.event)
ambiguous = reader.observe(demos["d02"])
show_clip(demos["d02"])
show_answer(ambiguous.result["phase_1"])

## Python's part

The allowed graph contains combat → rewards → map → combat, a pause-menu detour from any known phase, and menu → combat or map. Self-transitions are legal. A visible but unsupported jump, such as rewards → combat, is evidence to review; Python never rewrites it into a legal path. This graph belongs to the invented game and must change for another game.

A [confidence gate](../../docs/glossary.md#confidence-gate) is the rule that withholds a result below a chosen score. The candidate thresholds are the distinct minimum confidences observed on validation clips that already pass the fixed time, known-phase and graph checks. The objective is maximum accepted count subject to perfect validation event accuracy, implemented by selecting the lowest threshold with `target_accuracy=1.0`. A [gold label](../../docs/glossary.md#gold-label) is the authored expected phase sequence, used only for this selection and later scoring. [Accuracy](../../docs/glossary.md#accuracy) is the fraction of predictions matching those labels.

Freeze the selected threshold before looking at test answers. If no validation candidate meets the objective, selection raises rather than silently weakening it. Structural reviews do not get made-up confidence values. No valid prefix of a rejected clip is logged: history updates are atomic for this observation window.

In [ ]:
val_examples = select_split(examples, "validation")
val_observations = [reader.observe(e) for e in val_examples]
validation_pool = [
    (o.event == labels[e.id]["event"], o.confidence)
    for e, o in zip(val_examples, val_observations, strict=True)
    if helpers.structural_outcome(e.fields, o).outcome == helpers.ACCEPTED
]
threshold = select_confidence_threshold(
    [correct for correct, _ in validation_pool],
    [confidence for _, confidence in validation_pool],
    target_accuracy=1.0,
)
print(
    f"Validation N={len(val_examples)}, eligible N={len(validation_pool)}, frozen threshold={threshold:.4f}{selection}{check}"
)
print(
    f"Validation candidate thresholds: {sorted(set(round(c, 4) for _, c in validation_pool))}{selection}{check}"
)

demo_outcomes, demo_actions, demo_reviews = helpers.apply_outcomes(
    [demo, demos["d02"]], [demo_observation, ambiguous], threshold
)
print("Demo outcomes:", [(o.outcome, o.reason) for o in demo_outcomes])
print("Simulated ActionLog:", json.dumps(demo_actions.to_dicts(), indent=2))
print("ReviewQueue:", json.dumps(demo_reviews.to_dicts(), indent=2))

Each accepted change is bounded by the two selected capture timestamps: it happened after one observation and was visible by the next. These intervals are not exact event times. Sampling can miss a brief menu that opens and closes between captures, so a stable sampled path does not prove uninterrupted combat. The log records observations, not commands to a game.

## Evaluation

The frozen rule is compared with a deterministic pixel baseline on the same held-out clips. The baseline counts exact pixels in four fixed background colors and names the most common color if it covers at least 25% of a frame; otherwise it returns `uncertain`. It uses no labels, OCR, file-name meaning or model response. Both methods use the same sampling and graph checks. The baseline has no confidence estimate or fitted threshold. Alternate map/reward themes deliberately expose its limited color assumption.

**Event accuracy** means exact agreement on the complete collapsed phase path, including direction and intermediate changes, across the 11 clips whose capture metadata is valid. It also includes paths containing `other` or `uncertain`. The three capture failures have no inferable event and are excluded from raw event accuracy, while remaining in the 14-clip policy denominator.

[Coverage](../../docs/glossary.md#coverage) is the fraction of all clips accepted; [risk](../../docs/glossary.md#risk) is the error fraction among accepted clips. `evaluate_outcomes` uses the policy's actual accepted flags. Deferrals count toward review, never coverage. The test split is reported once, after selection; live unrecorded exploration prints no held-out metrics.

In [ ]:
val_outcomes, _, val_reviews = helpers.apply_outcomes(val_examples, val_observations, threshold)
val_metrics = evaluate_outcomes(
    [o.outcome == helpers.ACCEPTED for o in val_outcomes],
    [o.event == labels[e.id]["event"] for e, o in zip(val_examples, val_observations, strict=True)],
)
print(
    f"Validation N={val_metrics.n_total}: accepted={val_metrics.n_answered}, review={len(val_reviews)}, coverage={val_metrics.coverage:.4f}, accepted accuracy={val_metrics.accuracy:.4f}{selection}{check}"
)

In [ ]:
test_examples = select_split(examples, "test")
test_observations = [reader.observe(e) for e in test_examples]
test_outcomes, test_actions, test_reviews = helpers.apply_outcomes(
    test_examples, test_observations, threshold
)
baseline_observations = [helpers.pixel_baseline(e.fields, Path(".")) for e in test_examples]
baseline_outcomes = [
    helpers.structural_outcome(e.fields, o)
    for e, o in zip(test_examples, baseline_observations, strict=True)
]

if backend.mode == "live":
    print("Live exploration: record the answers before reporting held-out metrics.")
else:
    for name, observations, outcomes in (
        ("Multiframe Choice", test_observations, test_outcomes),
        ("Pixel baseline", baseline_observations, baseline_outcomes),
    ):
        valid_pairs = [
            (e, o)
            for e, o in zip(test_examples, observations, strict=True)
            if helpers.preflight(e.fields) is None
        ]
        raw_accuracy = accuracy(
            [labels[e.id]["event"] for e, _ in valid_pairs], [o.event for _, o in valid_pairs]
        )
        metrics = evaluate_outcomes(
            [o.outcome == helpers.ACCEPTED for o in outcomes],
            [
                o.event == labels[e.id]["event"]
                for e, o in zip(test_examples, observations, strict=True)
            ],
        )
        print(f"{name}: event accuracy={raw_accuracy:.4f}, N={len(valid_pairs)} valid clips{check}")
        print(
            f"{name}: N={metrics.n_total}, accepted={metrics.n_answered}, review={metrics.n_total - metrics.n_answered}, coverage={metrics.coverage:.4f}, abstention={1 - metrics.coverage:.4f}{check}"
        )
        print(
            f"{name}: accepted accuracy={metrics.accuracy:.4f}, risk={metrics.risk:.4f}, N={metrics.n_answered} accepted clips{check}"
        )
    print(f"Test per-clip audit, N={len(test_examples)}{check}")
    for e, o, routed, base in zip(
        test_examples, test_observations, test_outcomes, baseline_observations, strict=True
    ):
        print(
            e.id,
            "gold=",
            labels[e.id]["event"],
            "choice=",
            o.event,
            "policy=",
            routed.outcome,
            "baseline=",
            base.event,
            check,
        )
    print(
        "Simulated test history entries:",
        len(test_actions),
        "| pending review entries:",
        len(test_reviews),
        check,
    )

The frame confusion matrix below is a diagnostic view of the same 11 valid test clips (33 frame judgments). It does not replace event accuracy: one incorrect intermediate frame can corrupt an otherwise correct event path.

In [ ]:
if backend.mode != "live":
    valid = [(e, o) for e, o in zip(test_examples, test_observations, strict=True) if o.phases]
    matrix = confusion_matrix(
        [phase for e, _ in valid for phase in labels[e.id]["phases"]],
        [phase for _, o in valid for phase in o.phases],
        labels=list(helpers.OPTIONS),
    )
    print(f"Test frame matrix: N={len(valid) * 3} frames in {len(valid)} clips{check}")
    print("Columns:", list(matrix.labels), check)
    for label, counts in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
        print(label, counts, check)
    display(plot_confusion_matrix(matrix, title=f"33 test frame judgments{check}"))

### A permitted but wrong transition

The authored `t09` response says combat → menu at high confidence while the pixels show combat → rewards. Both edges are allowed, so graph checks cannot distinguish this mistake. The frozen policy accepts it. The displayed log entry therefore remains wrong; the notebook does not repair it using its gold label. A validation threshold is not a guarantee about held-out observations.

In [ ]:
wrong_index = next(i for i, e in enumerate(test_examples) if e.id == "t09")
show_clip(test_examples[wrong_index])
show_answer(test_observations[wrong_index].result["phase_2"])
print("Policy decision:", test_outcomes[wrong_index].outcome)
print("Observed event:", test_outcomes[wrong_index].event)
print("Review reasons:", [entry["reason"] for entry in test_reviews.to_dicts()])

## What was and was not measured

The cell below prints the run mode, the returned model and capture date when applicable, and N for each split. Synthetic values check only this pipeline. No live inference was performed for the committed notebook. Token usage, inference latency and cost remain `None`; local replay duration is not inference latency. A recorded run may retain actual returned usage, but absent timing or billing evidence stays unknown.

The clips share a drawing vocabulary and game rules. Independent scene seeds, file bytes and pixels across splits prevent exact image reuse; they do not make the test a realistic benchmark of another game. The frame cap can miss short-lived events, and the fixed graph can reject legitimate sequences in a different game. An unrecorded live run is exploration and has no reportable held-out result.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(val_examples)} validation, {len(test_examples)} test, 2 unscored demos{check}")
else:
    print(f"Provenance: {backend.mode}; returned model: {backend.model}")
    print(
        "Captured:", ", ".join(getattr(backend, "recorded_dates", ())) or "unrecorded exploration"
    )
    print(f"N: {len(val_examples)} validation, {len(test_examples)} test, 2 unscored demos")
print("Accounting:", run_record(demo_observation.result, mode=backend.mode))
print("Logical requests:", reader.requests, "| batched phase judgments:", 3 * reader.requests)
print("Synthetic source images:", len(list(Path("assets").glob("*.png"))))

## Next steps

Capture your own ordered screenshots and record answers with the image-aware request triples in the README. Keep capture metadata separate from phase labels, select a graph and threshold on validation, and reserve fresh test clips before changing the questions. Add short pause detours and low-frame-rate clips to measure what the sampling strategy misses.

For other bounded decisions, see [sentiment classification](../01-sentiment-classification/) for confidence gating and [failure recovery selection](../43-failure-recovery-selection/) for explicit recovery decisions. See the [vision helpers](../../docs/vision.md) for timestamped images and the [live recorder](../../docs/live.md#recorder) for genuine inference provenance.